In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
WEIGHTS_DIR = "gravnet_binary_classifier_faser_final"   # folder under get_weights_path()
RUN         = 20000
GPU         = "cuda:0"
NUM_EVENTS  = None      # None → load all events per chunk
SAVE_CACHE  = False     # set True to save inference results to disk

CLASS_NAMES      = ["background", "primary_EM_e"]
NUM_NODE_CLASSES = 2
# Shards used for Stage-1 classifier training — must match the training script.
CHUNKS = list(range(60, 100))
# Set to the Exp 3 (_vertexdist) weights dir to enable Exp2 vs Exp3 ROC comparison.
COMPARISON_WEIGHTS_DIR = "gravnet_binary_classifier_faser_vertexdist_final"   # e.g. "gravnet_binary_classifier_faser_vertexdist_final"
RUN_ROC_COMPARISON    = True   # set True (and set COMPARISON_WEIGHTS_DIR) to run Exp2 vs Exp3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
import torch
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_fscore_support, classification_report,
)
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

from analysis.gravnet.model import NeutrinoGravNetNodesFaser
from analysis.utils.utils import get_torch_path, get_weights_path, get_figures_path

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.2)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'figure.dpi': 350,
})

COLORS    = ["#4477AA", "#EE6677"]
MARKER_KW = dict(markersize=4, markerfacecolor='white', markeredgewidth=1.2)
C1 = "#353D4C"   # train
C2 = "#E17883"   # val
C3 = "#5691D9"   # recall

device       = torch.device(GPU if torch.cuda.is_available() else "cpu")
weights_path = get_weights_path() / WEIGHTS_DIR
torch_path   = get_torch_path()
figures_path = get_figures_path() / "gravnet_binary_classifier" / WEIGHTS_DIR
figures_path.mkdir(parents=True, exist_ok=True)

print(f"Device  : {device}")
print(f"Weights : {weights_path}")
print(f"Figures : {figures_path}")

__Training curves__

__Load data (held-out validation set)__

In [ ]:
# ── Early inference cache check ──────────────────────────────────────────────
# If a cache exists, load y_true/y_pred/y_prob now and skip data loading +
# model inference entirely. Re-run c4_load_data manually if you need val_dataset
# (event displays, per-event analysis).
_cache_candidates = sorted(figures_path.glob("infer_cache_*.npz"))
CACHE_LOADED = False
if _cache_candidates:
    _cp = _cache_candidates[-1]
    print(f"Cache found: {_cp.name}  — skipping data load and inference.")
    _c = np.load(_cp)
    y_true = _c["y_true"]; y_pred = _c["y_pred"]; y_prob = _c["y_prob"]
    CACHE_LOADED = True
    print(f"Loaded {len(y_true):,} nodes.  Overall accuracy: {(y_true == y_pred).mean():.4f}")
else:
    print("No cache found — will load data and run inference.")


In [ ]:
if not CACHE_LOADED:
    def get_str_from_run(run):
        return ["nue", "num", "nut", "nun"][run % 4]

    run_str  = get_str_from_run(RUN)
    run_path = torch_path / f"{RUN}/pointnetpp_faser_all_events"

    chunk_files = sorted(run_path.glob(f"{run_str}_*.pt"))
    chunk_files = [f for f in chunk_files
                   if "_particle_prob" not in f.stem and "_binary_prob" not in f.stem]
    chunk_files = [f for f in chunk_files if int(f.stem.split("_")[1]) in set(CHUNKS)]
    print(f"Run {RUN} ({run_str}): {len(chunk_files)} chunks in shards {CHUNKS[0]}–{CHUNKS[-1]}")

    dataset = []
    for chunk_file in chunk_files:
        chunk_data = torch.load(chunk_file, weights_only=False)
        if NUM_EVENTS is not None:
            chunk_data = chunk_data[:NUM_EVENTS]
        dataset.extend(chunk_data)
        print(f"  {chunk_file.name}: {len(chunk_data)} events")

    print(f"\nTotal events loaded: {len(dataset)}")

    _, val_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
    print(f"Val set            : {len(val_dataset)} events")

    from torch_geometric.loader import DataLoader
    if "_vertexdist" in WEIGHTS_DIR:
        print("Augmenting val set with ground-truth vertex distance.")
        for data in val_dataset:
            vertex_dist = torch.norm(
                data.pos - data.true_pos_centered.unsqueeze(0), dim=1, keepdim=True
            )
            data.x = torch.cat([data.x, vertex_dist], dim=1)

    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
else:
    print("Data load skipped (cache loaded). Run this cell manually if you need val_dataset.")


__Load model and run inference__

In [ ]:
ckpt = torch.load(weights_path / "best_model.pt", map_location=device, weights_only=False)
cfg  = ckpt.get("model_config", {})

input_dim = 2 if "_vertexdist" in WEIGHTS_DIR else 1

model = NeutrinoGravNetNodesFaser(
    input_dim=input_dim, num_node_classes=NUM_NODE_CLASSES, faser_dim=5,
    n_gravstack=cfg.get("n_gravstack", 3),
    out_channels=cfg.get("out_channels", 16),
    n_feature_transform=cfg.get("n_feature_transform", 16),
    k=cfg.get("k", 12),
).to(device)
model.load_state_dict(ckpt["model_state_dict"])

print(f"Checkpoint epoch : {ckpt['epoch'] + 1}")
print(f"Val loss         : {ckpt.get('val_loss', float('nan')):.4f}")
print(f"Val recall pEM   : {ckpt.get('val_recall_prim_EM', float('nan')):.4f}")
print(f"Parameters       : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

if CACHE_LOADED:
    print("\nInference skipped — y_true/y_pred/y_prob already loaded from cache.")
else:
    _ep        = ckpt['epoch']
    _vl        = ckpt.get('val_loss', 0.0)
    cache_path = figures_path / f"infer_cache_ep{_ep}_vl{_vl:.4f}.npz"

    if cache_path.exists():
        print(f"\nLoading inference from cache: {cache_path.name}")
        _c     = np.load(cache_path)
        y_true = _c["y_true"]; y_pred = _c["y_pred"]; y_prob = _c["y_prob"]
        print(f"Loaded {len(y_true):,} nodes.")
    else:
        print("\nRunning inference...")
        model.eval()
        all_targets, all_predictions, all_probabilities = [], [], []
        with torch.no_grad():
            for data in tqdm(val_loader, desc="Inference"):
                data = data.to(device)
                if data.x.size(0) == 0:
                    continue
                out  = model(data.x, data.pos, data.batch, data.x_faser)
                prob = torch.softmax(out, dim=1)
                pred = out.argmax(dim=1)
                binary_label = (data.pdg_label == 2).long()
                all_targets.append(binary_label.cpu().numpy())
                all_predictions.append(pred.cpu().numpy())
                all_probabilities.append(prob.cpu().numpy())
        y_true = np.concatenate(all_targets)
        y_pred = np.concatenate(all_predictions)
        y_prob = np.concatenate(all_probabilities)
        if SAVE_CACHE:
            np.savez_compressed(cache_path, y_true=y_true, y_pred=y_pred, y_prob=y_prob)
            print(f"Saved inference cache: {cache_path.name}")

    print(f"\nNodes evaluated : {len(y_true):,}")
    print(f"Overall accuracy: {(y_true == y_pred).mean():.4f}")


__Per-class metrics__

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=[0, 1], zero_division=0
)
print(f"{'Class':<20}  {'Precision':>10}  {'Recall':>10}  {'F1':>10}  {'Support':>12}")
print("-" * 68)
for name, p, r, f, s in zip(CLASS_NAMES, precision, recall, f1, support):
    print(f"{name:<20}  {p:>10.4f}  {r:>10.4f}  {f:>10.4f}  {s:>12,}")


__Confusion matrices__

In [ ]:
cm           = confusion_matrix(y_true, y_pred)
cm_norm_row  = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm_col  = cm.astype(float) / cm.sum(axis=0, keepdims=True)

FS = 14

def make_annot(cm_raw, cm_norm):
    annot = np.empty_like(cm_raw, dtype=object)
    for i in range(cm_raw.shape[0]):
        for j in range(cm_raw.shape[1]):
            annot[i, j] = f"{cm_norm[i, j]:.2f}\n({cm_raw[i, j]:,})"
    return annot

# Row-normalised (recall)
fig, ax = plt.subplots(figsize=(5, 4))
hm = sns.heatmap(cm_norm_row, annot=make_annot(cm, cm_norm_row), fmt="", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            vmin=0, vmax=1, cbar_kws={"label": "Recall (row fraction)"}, ax=ax,
            annot_kws={"size": FS})
ax.set_xlabel("Predicted", fontsize=FS); ax.set_ylabel("True", fontsize=FS)
ax.tick_params(labelsize=FS)
hm.collections[0].colorbar.set_label("Recall (row fraction)", fontsize=FS)
hm.collections[0].colorbar.ax.tick_params(labelsize=FS)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(figures_path / "confusion_matrices_recall.png", dpi=350, bbox_inches="tight")
plt.show()

# Column-normalised (precision)
fig, ax = plt.subplots(figsize=(5, 4))
hm = sns.heatmap(cm_norm_col, annot=make_annot(cm, cm_norm_col), fmt="", cmap="Greens",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            vmin=0, vmax=1, cbar_kws={"label": "Precision (col fraction)"}, ax=ax,
            annot_kws={"size": FS})
ax.set_xlabel("Predicted", fontsize=FS); ax.set_ylabel("True", fontsize=FS)
ax.tick_params(labelsize=FS)
hm.collections[0].colorbar.set_label("Precision (col fraction)", fontsize=FS)
hm.collections[0].colorbar.ax.tick_params(labelsize=FS)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(figures_path / "confusion_matrices_precision.png", dpi=350, bbox_inches="tight")
plt.show()


__Softmax probability distributions per true class__

For each true class, histogram of $P(\text{class})$ split by correct vs incorrect predictions.

__Prediction confidence distribution__

Confidence = $P(\text{predicted class})$, always $\geq 0.5$. Split by correct vs incorrect predictions.

In [ ]:
bins = np.linspace(0, 1, 31)
XLABEL = ["P(background)", "P(primary EM)"]

for true_idx, true_name in enumerate(CLASS_NAMES):
    fig, ax = plt.subplots(figsize=(8, 4))

    mask         = y_true == true_idx
    prob_col     = y_prob[mask, true_idx]
    correct_mask = y_pred[mask] == true_idx
    n_total      = int(mask.sum())

    ax.hist(prob_col[correct_mask],  bins=bins, alpha=0.4,
            color=COLORS[0],
            weights=np.ones(int(correct_mask.sum())) / n_total,
            label=f"Correct  ({correct_mask.sum():,})")
    ax.hist(prob_col[~correct_mask], bins=bins, alpha=0.4,
            color="#999999",
            weights=np.ones(int((~correct_mask).sum())) / n_total,
            label=f"Incorrect ({(~correct_mask).sum():,})")

    ax.set_xlabel(XLABEL[true_idx], fontsize=14)
    ax.set_ylabel("Node fraction per bin", fontsize=14)
    ax.tick_params(labelsize=14)
    ax.legend(fontsize=14)
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(figures_path / f"probability_distributions_v2_{'ab'[true_idx]}.png",
                dpi=350, bbox_inches="tight")
    plt.show()


__ROC curve__

__Optimal threshold (Youden index)__

Finds the threshold maximising $J = \text{TPR} - \text{FPR}$, i.e. the point on the ROC curve closest to the top-left corner.

__Event displays — best & worst by node accuracy (2-D)__

Three projections ($xz$, $xy$, $yz$) for the N best and N worst validation events ranked by per-event node accuracy.
Correct predictions: small filled circles, low opacity. Misclassified nodes: diamonds — **fill = predicted class, black edge** — so colour tells you what the node was wrongly called.

__Single event — interactive 3-D display (Plotly)__

Set `EVENT_IDX` to any val-dataset index. Rotate/zoom in the browser.
Colour = predicted class (blue = background, green = primary_EM_e); crosses = misclassified; hover shows $P(\text{primary\_EM\_e})$.

__Physics characterisation — what drives model performance?__

Augment per-event records with physics quantities, then investigate where the model struggles.

__Per-event accuracy vs physics quantities__

Each point is one val event. The black line is a **binned median**: events are sorted by the x-variable, divided into equal-count bins, and the median accuracy in each bin is plotted — a non-parametric way to show the trend without assuming a functional form.

__Spatial distribution of misclassified nodes (pooled across all val events)__

2-D histograms of misclassified node positions. Left: background nodes called primary_EM_e. Right: primary_EM_e nodes called background. Reveals *where* in the detector the decision boundary breaks down.

__PDG label breakdown of misclassified nodes__

For background nodes wrongly called primary_EM_e: which original particle types are being confused? Tells us whether the model struggles with secondary electrons (which genuinely look EM-like) vs hadronic hits vs muon hits.

In [ ]:
# Build all_pdg / all_pred / all_true without a second inference pass.
# val_loader has shuffle=False so node order matches val_dataset order.
PDG_NAMES = {0: "other/hadronic", 1: "secondary_e", 2: "primary_EM_e", 3: "muon"}
all_pdg  = np.concatenate([data.pdg_label.numpy() for data in val_dataset])
all_pred = y_pred   # from c5_inference
all_true = y_true   # from c5_inference
print(f"Pooled {len(all_pdg):,} nodes across {len(val_dataset)} val events.")

In [ ]:
# ── P(pEM) distribution per background PDG class ─────────────────────────────
PDG_BG_COLORS = {0: "#4B9BEB", 1: "#FC5C5C", 3: "#3EC679"}
p_thresh = 0.015
bins_left  = np.linspace(0, p_thresh, 25)
bins_right = np.linspace(p_thresh, 1,  25)

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(10, 6),
                                  gridspec_kw={"width_ratios": [1, 3], "wspace": 0.1})
for p in [3, 0, 1]:
    mask  = all_pdg == p
    n     = int(mask.sum())
    label = f"{PDG_NAMES[p]}  (n = {n:,})"
    kw = dict(density=True, color=PDG_BG_COLORS[p], histtype='step', lw=1.6, alpha=0.8)
    ax_l.hist(y_prob[mask, 1], bins=bins_left,  **kw)
    ax_r.hist(y_prob[mask, 1], bins=bins_right, label=label, **kw)

ax_l.set_xlabel(r"$P(\mathrm{primary\_EM\_e})$", fontsize=10)
ax_l.set_ylabel("Density", fontsize=10)
ax_l.set_title(f"$P < {p_thresh}$", fontsize=9)
ax_l.spines[["top", "right"]].set_visible(False)

ax_r.set_xlabel(r"$P(\mathrm{primary\_EM\_e})$", fontsize=10)
ax_r.set_ylabel("Density", fontsize=10)
ax_r.set_title(f"$P > {p_thresh}$", fontsize=9)
ax_r.legend(fontsize=8, frameon=False, loc='upper right')
ax_r.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(figures_path / "pdg_probability_distributions.png", dpi=350, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion rate vs inelasticity y, per background PDG class ───────────────
PDG_BG_COLORS = {0: "#999999", 1: COLORS[1], 3: COLORS[0]}

_node_counts_cr = [data.num_nodes for data in val_dataset]
_ev_inel_cr     = np.array([float(data.E_roe) / float(data.E_nu)
                             if float(data.E_nu) > 0 else np.nan
                             for data in val_dataset])
_node_inel_cr   = np.repeat(_ev_inel_cr, _node_counts_cr)

y_bins   = np.linspace(0, 1, 11)
bin_ctrs = 0.5 * (y_bins[:-1] + y_bins[1:])

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.15, zorder=0)

for p in [3, 0, 1]:
    mask_pdg = all_pdg == p
    rates = []
    for ylo, yhi in zip(y_bins[:-1], y_bins[1:]):
        bin_mask = (_node_inel_cr >= ylo) & (_node_inel_cr < yhi) & mask_pdg
        n_bin  = int(bin_mask.sum())
        n_conf = int((bin_mask & (all_pred == 1)).sum())
        rates.append(n_conf / n_bin if n_bin > 0 else np.nan)
    ax.plot(bin_ctrs, rates, marker='o', markersize=4, lw=1.1,
            linestyle='--', color=PDG_BG_COLORS[p], label=PDG_NAMES[p], zorder=3)

ax.set_xlabel("Inelasticity $y$", fontsize=14)
ax.set_ylabel("Confusion rate", fontsize=14)
ax.legend(fontsize=14, frameon=False)
ax.tick_params(axis='both', labelsize=14)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(figures_path / "pdg_confusion_rate_vs_y.png", dpi=350, bbox_inches='tight')
plt.show()


__Secondary electron analysis__

For each true PDG class: how often does the model predict primary_EM_e? For secondary electrons (PDG=1) this is the key question — they are EM-like background nodes and the hardest to separate. Right panel shows the purity of primary_EM_e predictions (what fraction are actually primary_EM_e vs contamination).

__Inelasticity $y \\approx 0.5$ \u2014 worst-performing events (Task A)__

Select events with $y \\in [0.45,\\,0.55]$ and rank by per-event primary\_EM\_e F1.
For each event: true labels vs predicted labels ($xz$ projection), misclassified nodes
marked with diamonds, and a boundary/distributed confusion classification.

In [ ]:
# ── Exp 2 vs Exp 3 classifier comparison ─────────────────────────────────────
# Set COMPARISON_WEIGHTS_DIR in the config cell to the Exp 3 weights directory.
# Loads Exp 3, runs inference on the same val nodes, then:
#   • overlays both ROC curves on one plot
#   • computes bootstrap 95% CI on ΔAUC (paired node-level resampling)

if not RUN_ROC_COMPARISON:
    print("ROC comparison skipped — set RUN_ROC_COMPARISON = True to enable.")
else:
    import torch
    from sklearn.metrics import roc_curve, auc as sk_auc

    comp_weights_path = get_weights_path() / COMPARISON_WEIGHTS_DIR
    _ckpt_comp = torch.load(comp_weights_path / "best_model.pt",
                            map_location=device, weights_only=False)
    _cfg_comp  = _ckpt_comp.get("model_config", {})
    _vertexdist_comp = "_vertexdist" in COMPARISON_WEIGHTS_DIR

    comp_input_dim = 2 if _vertexdist_comp else 1
    comp_model = NeutrinoGravNetNodesFaser(
        input_dim=comp_input_dim, num_node_classes=NUM_NODE_CLASSES, faser_dim=5,
        n_gravstack=_cfg_comp.get("n_gravstack", 3),
        out_channels=_cfg_comp.get("out_channels", 16),
        n_feature_transform=_cfg_comp.get("n_feature_transform", 16),
        k=_cfg_comp.get("k", 12),
    ).to(device)
    comp_model.load_state_dict(_ckpt_comp["model_state_dict"])
    comp_model.eval()
    print(f"Loaded comparison model: {COMPARISON_WEIGHTS_DIR}")
    print(f"  epoch={_ckpt_comp['epoch']+1}  vertexdist={_vertexdist_comp}")

    # Run inference — augment with vertex distance on-the-fly if needed
    all_prob_comp = []
    with torch.no_grad():
        for data in tqdm(val_loader, desc="Comp inference"):
            data = data.to(device)
            if data.x.size(0) == 0:
                continue
            x_in = data.x
            if _vertexdist_comp:
                vdist = torch.norm(
                    data.pos - data.true_pos_centered.view(-1, 3)[data.batch], dim=1, keepdim=True
                )
                x_in = torch.cat([x_in, vdist], dim=1)
            out  = comp_model(x_in, data.pos, data.batch, data.x_faser)
            prob = torch.softmax(out, dim=1)
            all_prob_comp.append(prob.cpu().numpy())
    y_prob_comp = np.concatenate(all_prob_comp)
    print(f"Comp inference done: {len(y_prob_comp):,} nodes")

    # ── ROC curves overlaid ──────────────────────────────────────────────────
    fpr2, tpr2, _ = roc_curve(y_true, y_prob[:, 1])
    fpr3, tpr3, _ = roc_curve(y_true, y_prob_comp[:, 1])
    auc2 = sk_auc(fpr2, tpr2)
    auc3 = sk_auc(fpr3, tpr3)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr2, tpr2, color=COLORS[1], lw=1.8,
            label=f"Exp 2 — binary classifier  (AUC = {auc2:.3f})")
    ax.plot(fpr3, tpr3, color=COLORS[0], lw=1.8, linestyle='--',
            label=f"Exp 3 — + oracle vertex dist  (AUC = {auc3:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC comparison: Exp 2 vs Exp 3")
    ax.legend(fontsize=9, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(figures_path / "roc_comparison.png", dpi=350, bbox_inches="tight")
    plt.show()

    # ── Bootstrap 95% CI on ΔAUC (subsampled node resampling) ───────────────
    # Subsample N_BOOT nodes per iteration — roc_curve is O(n log n) on CPU
    # and resampling the full val set would take hours.
    N_BOOT = 50_000
    rng = np.random.default_rng(42)
    n   = len(y_true)
    delta_aucs = []
    for _ in tqdm(range(500), desc="Bootstrap CI"):
        idx   = rng.integers(0, n, N_BOOT)
        a2    = sk_auc(*roc_curve(y_true[idx], y_prob[idx, 1])[:2])
        a3    = sk_auc(*roc_curve(y_true[idx], y_prob_comp[idx, 1])[:2])
        delta_aucs.append(a3 - a2)
    d_med       = float(np.median(delta_aucs))
    d_lo, d_hi  = np.percentile(delta_aucs, [2.5, 97.5])
    print(f"\nAUC Exp 2 : {auc2:.4f}")
    print(f"AUC Exp 3 : {auc3:.4f}")
    print(f"\u0394AUC (Exp3 \u2212 Exp2): {d_med:+.4f}  95% CI [{d_lo:+.4f}, {d_hi:+.4f}]")
    if d_lo > 0:
        print("\u2192 CI excludes zero: oracle vertex distance significantly improves AUC.")
    else:
        print("\u2192 CI spans zero: improvement not statistically significant at 95%.")
    print("Saved: roc_comparison.png")


In [ ]:
# ── Save comprehensive classifier stats to report_stats.json ─────────────────
import json as _json
import datetime as _dt
import numpy as _np
from sklearn.metrics import (confusion_matrix as _cm,
                             precision_recall_fscore_support as _prfs,
                             roc_curve as _roc_curve)
from sklearn.metrics import auc as _sk_auc

# Confusion matrix (absolute counts)
_cm_raw = _cm(y_true, y_pred)
_TP = int(_cm_raw[1, 1]); _FP = int(_cm_raw[0, 1])
_TN = int(_cm_raw[0, 0]); _FN = int(_cm_raw[1, 0])

# Per-class metrics
_prec, _rec, _f1, _sup = _prfs(y_true, y_pred, labels=[0, 1], zero_division=0)

# ROC
_fpr, _tpr, _thresh = _roc_curve(y_true, y_prob[:, 1])
_auc_val = float(_sk_auc(_fpr, _tpr))

# Optimal threshold (Youden J)
_J   = _tpr - _fpr
_opt = int(_np.argmax(_J))

# Per-event accuracy vs inelasticity y
# (requires that val_dataset has been processed and per-event arrays are available)
_per_bin_y_acc = []
try:
    # Reconstruct per-event accuracy and inelasticity from val_dataset + node-level arrays
    # node counts per event: data.num_nodes
    _node_counts = [data.num_nodes for data in val_dataset]
    _splits      = _np.cumsum(_node_counts)[:-1]
    _y_true_ev   = _np.split(y_true, _splits)
    _y_pred_ev   = _np.split(y_pred, _splits)
    _ev_acc      = _np.array([(yt == yp).mean() for yt, yp in zip(_y_true_ev, _y_pred_ev)])
    _ev_inel     = _np.array([float(data.E_roe / max(data.E_nu, 1e-6)) for data in val_dataset])
    _y_bins      = _np.linspace(0, 1, 11)
    for _ymin, _ymax in zip(_y_bins[:-1], _y_bins[1:]):
        _mask = (_ev_inel >= _ymin) & (_ev_inel < _ymax)
        _n    = int(_mask.sum())
        if _n < 5:
            continue
        _per_bin_y_acc.append({
            "ymin": round(float(_ymin), 2), "ymax": round(float(_ymax), 2),
            "n_events": _n,
            "mean_accuracy": float(_ev_acc[_mask].mean()),
            "median_accuracy": float(_np.median(_ev_acc[_mask])),
            "std_accuracy": float(_ev_acc[_mask].std()),
        })
except Exception as _e:
    print(f"  per-event y-accuracy binning skipped: {_e}")


# Per-PDG confusion rates (uses all_pdg / all_pred from cell 256dd091)
_pdg_conf_rates = {}
for _p, _pkey in [(1, "secondary_e"), (0, "other_hadronic"), (3, "muon")]:
    _m_pdg        = all_pdg == _p
    _n_total_pdg  = int(_m_pdg.sum())
    _n_conf_pdg   = int(((all_pdg == _p) & (all_pred == 1)).sum())
    _pdg_conf_rates[_pkey] = {
        "n_total":    _n_total_pdg,
        "n_confused": _n_conf_pdg,
        "rate":       float(_n_conf_pdg / _n_total_pdg) if _n_total_pdg > 0 else 0.0,
    }
_stats_clf = {
    "meta": {
        "weights_dir":   WEIGHTS_DIR,
        "epoch":         int(ckpt["epoch"]) + 1,
        "val_loss":      float(ckpt.get("val_loss", float("nan"))),
        "n_val_nodes":   int(len(y_true)),
        "class_names":   CLASS_NAMES,
        "timestamp":     _dt.datetime.now().isoformat(),
    },
    "overall": {
        "accuracy":           float((y_true == y_pred).mean()),
        "auc":                _auc_val,
        "n_positive_true":    int((y_true == 1).sum()),
        "n_negative_true":    int((y_true == 0).sum()),
        "positive_class_frac": float((y_true == 1).mean()),
    },
    "per_class": {
        CLASS_NAMES[0]: {
            "precision": float(_prec[0]), "recall": float(_rec[0]),
            "f1": float(_f1[0]), "support": int(_sup[0]),
        },
        CLASS_NAMES[1]: {
            "precision": float(_prec[1]), "recall": float(_rec[1]),
            "f1": float(_f1[1]), "support": int(_sup[1]),
        },
    },
    "confusion_matrix": {
        "TP": _TP, "FP": _FP, "TN": _TN, "FN": _FN,
        "TPR": float(_TP / max(_TP + _FN, 1)),   # recall / sensitivity
        "TNR": float(_TN / max(_TN + _FP, 1)),   # specificity
        "FPR": float(_FP / max(_FP + _TN, 1)),
        "FNR": float(_FN / max(_FN + _TP, 1)),
    },
    "optimal_threshold": {
        "threshold":    float(_thresh[_opt]),
        "tpr":          float(_tpr[_opt]),
        "fpr":          float(_fpr[_opt]),
        "youden_j":     float(_J[_opt]),
    },
    "per_event_accuracy_vs_y": _per_bin_y_acc,
    "pdg_confusion_rates":    _pdg_conf_rates,
}

# ── Comparison result (Exp2 vs Exp3) if roc_comparison was run ───────────────
try:
    if RUN_ROC_COMPARISON and COMPARISON_WEIGHTS_DIR is not None and "y_prob_comp" in dir():
        _fpr3, _tpr3, _ = _roc_curve(y_true, y_prob_comp[:, 1])
        _auc3 = float(_sk_auc(_fpr3, _tpr3))

        # Bootstrap CI on ΔAUC (subsampled; same seed as roc_comparison cell)
        _N_BOOT = 50_000
        _rng2 = _np.random.default_rng(42)
        _n2   = len(y_true)
        _dauc = []
        for _ in range(500):
            _idx2 = _rng2.integers(0, _n2, _N_BOOT)
            _a2   = _sk_auc(*_roc_curve(y_true[_idx2], y_prob[_idx2, 1])[:2])
            _a3   = _sk_auc(*_roc_curve(y_true[_idx2], y_prob_comp[_idx2, 1])[:2])
            _dauc.append(_a3 - _a2)
        _dauc   = _np.array(_dauc)
        _d_med  = float(_np.median(_dauc))
        _d_lo, _d_hi = [float(x) for x in _np.percentile(_dauc, [2.5, 97.5])]

        _stats_clf["comparison_vs"] = {
            "comparison_weights_dir": COMPARISON_WEIGHTS_DIR,
            "auc_primary":            _auc_val,
            "auc_comparison":         _auc3,
            "delta_auc_median":       _d_med,
            "delta_auc_ci_lo":        _d_lo,
            "delta_auc_ci_hi":        _d_hi,
            "significant_at_95pct":   bool(_d_lo > 0),
        }
        print(f"  ΔAUC = {_d_med:+.4f}  [{_d_lo:+.4f}, {_d_hi:+.4f}]")
except Exception as _e:
    print(f"  comparison stats skipped: {_e}")

# ── Write to report_stats.json ────────────────────────────────────────────────
# Write this model's stats to its own file (avoids loading full report_stats.json)
_model_stats_file = get_weights_path() / WEIGHTS_DIR / "eval_stats.json"
_model_stats_file.write_text(_json.dumps(_stats_clf, indent=2))
# Also merge into report_stats.json, but stream-update to avoid loading huge file
_stats_file = get_weights_path() / "report_stats.json"
try:
    _existing = _json.loads(_stats_file.read_text()) if _stats_file.exists() else {}
    _existing[WEIGHTS_DIR] = _stats_clf
    _stats_file.write_text(_json.dumps(_existing, indent=2))
except Exception as _je:
    print(f"  report_stats.json merge skipped ({_je}); stats saved to {_model_stats_file}")
print(f"Saved classifier stats → {_stats_file}")
print(f"  AUC = {_auc_val:.4f}")
print(f"  F1 ({CLASS_NAMES[1]}) = {_f1[1]:.4f}  "
      f"precision={_prec[1]:.4f}  recall={_rec[1]:.4f}")